In [3]:
import pandas as pd
import numpy as np
import glob
import os
from typing import Dict, List, Tuple
import re

def extract_final_architecture_string(filepath: str) -> str:
    """Extract the final architecture string from architecture history file"""
    base_name = os.path.basename(filepath).replace('_architecture_history.txt', '')
    prefix = base_name.split('_')[0]
    last_number = prefix.split('.')[-1]
    final_architecture = []
    
    with open(filepath, 'r') as f:
        lines = f.readlines()
        final_section_start = -1
        for i, line in enumerate(lines):
            if line.strip() == "Final Architecture:":
                final_section_start = i
                break
                
        if final_section_start == -1:
            raise ValueError("No 'Final Architecture:' section found in file")
            
        for line in lines[final_section_start + 1:]:
            line = line.strip()
            if not line:
                break
                
            if "Layer" in line and ":" in line:
                parts = line.split(":")
                if len(parts) != 2:
                    continue
                    
                neurons_info = parts[1].strip()
                if "Not active" in neurons_info:
                    final_architecture.append('n')
                else:
                    try:
                        num_neurons = int(neurons_info.split()[0])
                        final_architecture.append(str(num_neurons))
                    except (ValueError, IndexError):
                        continue
    
    architecture_str = '_'.join(final_architecture)
    result = f"{prefix}.{last_number}.{architecture_str}"
    return result

def process_csv(file_path: str) -> pd.DataFrame:
    """Process a CSV file containing validation accuracy data"""
    df = pd.read_csv(file_path)
    df['value'] = df['value'].apply(lambda x: eval(x)[0])
    df = df.drop_duplicates(subset=['step'], keep='last')
    return df

def calculate_metrics(df: pd.DataFrame) -> Dict[str, float]:
    """Calculate various performance metrics from the DataFrame"""
    final_accuracy = df['value'].tail(10).mean()
    
    # Convergence Time (steps to reach within 1% of final value)
    final_val = df['value'].iloc[-1]
    if 'loss' in directory_path.lower():
        # For loss: when it drops to within 1% of final loss
        convergence_threshold = final_val * 1.01
        convergence_step = df[df['value'] <= convergence_threshold]['step'].iloc[0]
    else:
        # For accuracy: when it reaches 99% of final accuracy
        convergence_threshold = final_val * 0.99
        convergence_step = df[df['value'] >= convergence_threshold]['step'].iloc[0]
    
    # Training Stability (CV)
    cv = df['value'].std() / df['value'].mean()
    
    # Computing Time
    total_time = df['wall_time'].iloc[-1] - df['wall_time'].iloc[0]
    avg_time_per_step = total_time / len(df)
    
    return {
        'final_accuracy': final_accuracy,
        'convergence_time': convergence_step,
        'stability_cv': cv,
        'total_training_time': total_time,
        'avg_time_per_step': avg_time_per_step
    }

def aggregate_metrics(metrics_list: List[Dict[str, float]]) -> Dict[str, Tuple[float, float]]:
    """Calculate mean and std for each metric across multiple runs"""
    all_metrics = {}
    for metric in metrics_list[0].keys():
        values = [m[metric] for m in metrics_list]
        mean_val = np.mean(values)
        std_val = np.std(values)
        all_metrics[metric] = (mean_val, std_val)
    return all_metrics

def format_aggregate_metrics(senn_metrics: Dict[str, Tuple[float, float]], 
                           mlp_metrics: Dict[str, Tuple[float, float]], tag: str) -> None:
    """Print formatted comparison of metrics"""
    print(f"\nPerformance Metrics Comparison ({tag}):")
    print("-" * 80)
    print(f"{'Metric':<30} {'SENN':^22} {'MLP':^22}")
    print("-" * 80)
    
    # Determine metric name based on directory path
    metric_name = 'Loss' if 'loss' in directory_path.lower() else 'Accuracy'
    
    metrics_to_format = {
        f'Final {metric_name}': 'final_accuracy',
        'Convergence Time (steps)': 'convergence_time',
        'Training Stability (CV)': 'stability_cv',
        'Total Training Time (s)': 'total_training_time',
        'Avg Time per Step (s)': 'avg_time_per_step'
    }
    
    for display_name, metric_key in metrics_to_format.items():
        senn_mean, senn_std = senn_metrics[metric_key]
        mlp_mean, mlp_std = mlp_metrics[metric_key]
        
        senn_str = f"{senn_mean:.3f} ± {senn_std:.3f}"
        mlp_str = f"{mlp_mean:.3f} ± {mlp_std:.3f}"
        print(f"{display_name:<30} {senn_str:<22} {mlp_str:<22}")

def process_directory(directory_path: str) -> None:
    """Process all validation accuracy files in directory"""
    tag = directory_path.split('/')[-1]
    csv_files = glob.glob(os.path.join(directory_path, f'*_{tag}.csv'))
    
    if not csv_files:
        # print(f"No CSV files found in {directory_path}")
        return
    
    experiment_name = os.path.basename(os.path.dirname(directory_path))
    arch_base_path = os.path.join('/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/final_architectures', experiment_name)
    
    senn_metrics = []
    mlp_metrics = []
    base_to_arch = {}
    
    # First identify base experiments and their final architectures
    for csv_file in csv_files:
        base_name = os.path.basename(csv_file).replace(f'_{tag}.csv', '')
        parts = base_name.split('.')
        
        if len(parts) == 2:  # Base SENN experiment
            arch_file = os.path.join(arch_base_path, f'{base_name}_architecture_history.txt')
            if os.path.exists(arch_file):
                try:
                    final_arch = extract_final_architecture_string(arch_file)
                    base_to_arch[base_name] = final_arch
                    # Process SENN metrics
                    df = process_csv(csv_file)
                    senn_metrics.append(calculate_metrics(df))
                except Exception as e:
                    print(f"Error processing SENN file {base_name}: {e}")
    
    # Now process final architecture MLPs
    for csv_file in csv_files:
        base_name = os.path.basename(csv_file).replace(f'_{tag}.csv', '')
        if base_name in base_to_arch.values():  # Only process final architectures
            try:
                df = process_csv(csv_file)
                mlp_metrics.append(calculate_metrics(df))
            except Exception as e:
                print(f"Error processing MLP file {base_name}: {e}")
    
    if not senn_metrics or not mlp_metrics:
        print("No valid files found for either SENN or MLP")
        return
    
    # Calculate aggregate metrics
    senn_aggregate = aggregate_metrics(senn_metrics)
    mlp_aggregate = aggregate_metrics(mlp_metrics)
    
    # Print results
    # print(f"\nProcessed {len(senn_metrics)} SENN runs and {len(mlp_metrics)} MLP runs")
    format_aggregate_metrics(senn_aggregate, mlp_aggregate, tag=tag)

In [4]:
for ex in range(1, 3):
    print(f"\nProcessing Experiment {ex}...")
    for tag in ["loss", "validation loss", "training accuracy", "validation accuracy"]:
        directory_path = f"/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports/experiment{ex}/{tag}"
        process_directory(directory_path)


Processing Experiment 1...

Performance Metrics Comparison (loss):
--------------------------------------------------------------------------------
Metric                                  SENN                   MLP          
--------------------------------------------------------------------------------
Final Loss                     0.000 ± 0.000          0.000 ± 0.000         
Convergence Time (steps)       1951.633 ± 43.334      1930.367 ± 80.783     
Training Stability (CV)        16.818 ± 3.165         16.919 ± 3.395        
Total Training Time (s)        4064.238 ± 127.444     83.701 ± 6.232        
Avg Time per Step (s)          2.032 ± 0.064          0.042 ± 0.003         

Performance Metrics Comparison (validation loss):
--------------------------------------------------------------------------------
Metric                                  SENN                   MLP          
--------------------------------------------------------------------------------
Final Loss        

In [5]:
def print_value(directory_path: str, time_step: int) -> None:
    """Process all validation accuracy files and print mean ± std at given time step"""
    tag = directory_path.split('/')[-1]
    csv_files = glob.glob(os.path.join(directory_path, f'*_{tag}.csv'))
    
    if not csv_files:
        print(f"No CSV files found in {directory_path}")
        return
    
    experiment_name = os.path.basename(os.path.dirname(directory_path))
    arch_base_path = os.path.join('/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/final_architectures', experiment_name)
    
    senn_values = []
    mlp_values = []
    base_to_arch = {}
    
    # Process SENN files
    for csv_file in csv_files:
        base_name = os.path.basename(csv_file).replace(f'_{tag}.csv', '')
        parts = base_name.split('.')
        
        if len(parts) == 2:  # Base SENN experiment
            arch_file = os.path.join(arch_base_path, f'{base_name}_architecture_history.txt')
            if os.path.exists(arch_file):
                try:
                    final_arch = extract_final_architecture_string(arch_file)
                    base_to_arch[base_name] = final_arch
                    df = process_csv(csv_file)
                    value = df[df['step'] == time_step]['value'].values[0]
                    senn_values.append(value)
                except Exception as e:
                    print(f"Error processing SENN file {base_name}: {e}")
    
    # Process MLP files
    for csv_file in csv_files:
        base_name = os.path.basename(csv_file).replace(f'_{tag}.csv', '')
        if base_name in base_to_arch.values():
            try:
                df = process_csv(csv_file)
                value = df[df['step'] == time_step]['value'].values[0]
                mlp_values.append(value)
            except Exception as e:
                print(f"Error processing MLP file {base_name}: {e}")
    
    if senn_values and mlp_values:
        senn_mean, senn_std = np.mean(senn_values), np.std(senn_values)
        mlp_mean, mlp_std = np.mean(mlp_values), np.std(mlp_values)
        print(f"\nValues at step {time_step}:")
        print("-" * 80)
        print(f"{'Model':<30} {'Value':^22}")
        print("-" * 80)
        print(f"{'SENN':<30} {senn_mean:.3f} ± {senn_std:.3f}")
        print(f"{'MLP':<30} {mlp_mean:.3f} ± {mlp_std:.3f}")

In [6]:
tag = "validation accuracy"
ex = 2
time_step = 115
directory_path = f"/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports/experiment{ex}/{tag}"
# Call the function with the given directory_path and time_step
print_value(directory_path, time_step=time_step)


Values at step 115:
--------------------------------------------------------------------------------
Model                                  Value         
--------------------------------------------------------------------------------
SENN                           0.897 ± 0.008
MLP                            0.899 ± 0.010


## Expansion Analysis

In [7]:
import glob
import os
import numpy as np
from typing import List, Dict, Tuple
import re

def parse_architecture(section: List[str]) -> Dict[int, int]:
    """Parse a single architecture section to get layer sizes."""
    layer_sizes = {}
    for line in section:
        if "Layer" in line and ":" in line:
            try:
                layer_num = int(line.split("Layer")[1].split(":")[0].strip())
                if "Not active" in line:
                    layer_sizes[layer_num] = 0
                else:
                    neurons = int(line.split("neurons")[0].split(":")[-1].strip())
                    layer_sizes[layer_num] = neurons
            except (ValueError, IndexError):
                continue
    return layer_sizes

def determine_expansion_type(prev_arch: Dict[int, int], curr_arch: Dict[int, int]) -> str:
    """Determine if expansion was layer addition or neuron addition."""
    active_layers_prev = sum(1 for size in prev_arch.values() if size > 0)
    active_layers_curr = sum(1 for size in curr_arch.values() if size > 0)
    
    if active_layers_curr > active_layers_prev:
        return "layer"
    return "neuron"

def extract_expansion_details(filepath: str) -> List[Tuple[int, str]]:
    """Extract epochs and types of architecture changes."""
    expansions = []
    architectures = []
    current_section = []
    current_epoch = None
    
    with open(filepath, 'r') as f:
        lines = f.readlines()
        
        for line in lines:
            line = line.strip()
            
            if not line:
                if current_section:
                    arch = parse_architecture(current_section)
                    if current_epoch is not None:
                        architectures.append((current_epoch, arch))
                    current_section = []
                continue
                
            if "Initial Architecture:" in line or "Final Architecture:" in line:
                current_epoch = 0
                current_section = []
            elif "Architecture after epoch" in line:
                try:
                    current_epoch = int(line.split("epoch")[1].split(":")[0].strip())
                    current_section = []
                except (ValueError, IndexError):
                    continue
            
            current_section.append(line)
            
        # Process last section if exists
        if current_section:
            arch = parse_architecture(current_section)
            if current_epoch is not None:
                architectures.append((current_epoch, arch))
    
    # Determine expansion types by comparing consecutive architectures
    # Skip the initial architecture (index 0) when it has epoch 0
    start_idx = 1 if architectures and architectures[0][0] == 0 else 0
    
    for i in range(start_idx, len(architectures)):
        epoch = architectures[i][0]
        prev_arch = architectures[i-1][1]
        curr_arch = architectures[i][1]
        exp_type = determine_expansion_type(prev_arch, curr_arch)
        if epoch > 0:  # Only include non-zero epochs
            expansions.append((epoch, exp_type))
    
    return sorted(expansions, key=lambda x: x[0])

def calculate_parameters(architecture: Dict[int, int], input_size: int) -> int:
    """Calculate total parameters for MLP architecture."""
    active_layers = [(k, v) for k, v in sorted(architecture.items()) if v > 0]
    if not active_layers:
        return 0
        
    total_params = 0
    prev_size = input_size
    
    for _, curr_size in active_layers:
        # Weight parameters: prev_size * curr_size
        # Bias parameters: curr_size
        params = (prev_size * curr_size) + curr_size
        total_params += params
        prev_size = curr_size
        
    return total_params

def analyze_expansion_timings(directory_path: str, experiment: str = "ex1") -> None:
    """Analyze expansion timings and types across all architecture files."""
    input_size = 1 if experiment == "ex1" else 2  # ex1: regression, ex2: half moon
    """Analyze expansion timings and types across all architecture files."""
    arch_files = glob.glob(os.path.join(directory_path, '*architecture_history.txt'))
    
    if not arch_files:
        print("No architecture history files found")
        return
        
    expansion_timings: Dict[int, List[Tuple[int, str]]] = {}
    
    for file_path in arch_files:
        try:
            expansions = extract_expansion_details(file_path)
            
            for i, (epoch, exp_type) in enumerate(expansions):
                if i not in expansion_timings:
                    expansion_timings[i] = []
                expansion_timings[i].append((epoch, exp_type))
                
        except Exception as e:
            print(f"Error processing {os.path.basename(file_path)}: {e}")
            continue
    
    # Track architectures at each expansion point
    expansion_architectures: Dict[int, List[Dict[int, int]]] = {}
    
    for file_path in arch_files:
        current_section = []
        current_epoch = None
        current_arch = None
        last_expansion_num = -1
        
        with open(file_path, 'r') as f:
            lines = f.readlines()
            
            for line in lines:
                line = line.strip()
                
                if not line:
                    if current_section and current_epoch is not None:
                        arch = parse_architecture(current_section)
                        if current_arch is None or arch != current_arch:
                            current_arch = arch
                            last_expansion_num += 1
                            if last_expansion_num not in expansion_architectures:
                                expansion_architectures[last_expansion_num] = []
                            expansion_architectures[last_expansion_num].append(arch)
                    current_section = []
                    continue
                
                if "Architecture after epoch" in line:
                    try:
                        current_epoch = int(line.split("epoch")[1].split(":")[0].strip())
                        current_section = []
                    except (ValueError, IndexError):
                        continue
                        
                current_section.append(line)
    
    print("\nExpansion Timing Analysis:")
    print("-" * 100)
    print(f"Analyzed {len(arch_files)} architecture files")
    print("-" * 100)
    print(f"{'Expansion':<10} {'Mean Epoch':<12} {'Std Dev':<12} {'Count':<8} {'Neuron Add %':<12} {'Mean Params':<12}")
    print("-" * 100)
    
    for exp_num in sorted(expansion_timings.keys()):
        expansions = expansion_timings[exp_num]
        epochs = [e[0] for e in expansions]
        mean_epoch = np.mean(epochs)
        std_epoch = np.std(epochs)
        count = len(epochs)
        
        # Calculate percentage of neuron additions
        neuron_adds = sum(1 for _, type_ in expansions if type_ == "neuron")
        neuron_percent = (neuron_adds / count) * 100
        
        # Calculate mean parameters for this expansion
        if exp_num in expansion_architectures:
            params = [calculate_parameters(arch, input_size) for arch in expansion_architectures[exp_num]]
            mean_params = np.mean(params)
        else:
            mean_params = 0
            
        print(f"{f'#{exp_num+1}':<10} {f'{mean_epoch:.1f}':<12} {f'{std_epoch:.1f}':<12} "
              f"{count:<8} {f'{neuron_percent:.1f}%':<12} {f'{mean_params:.1f}':<12}")
    
    # Additional statistics
    all_files_expansions = [len(extract_expansion_details(f)) for f in arch_files]
    avg_expansions = np.mean(all_files_expansions)
    std_expansions = np.std(all_files_expansions)
    
    print("-" * 100)
    print(f"Average number of expansions per network: {avg_expansions:.1f} ± {std_expansions:.1f}")

In [8]:
ex = 1
directory_path = f"/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/final_architectures/experiment{ex}"

analyze_expansion_timings(directory_path, experiment=ex)


Expansion Timing Analysis:
----------------------------------------------------------------------------------------------------
Analyzed 30 architecture files
----------------------------------------------------------------------------------------------------
Expansion  Mean Epoch   Std Dev      Count    Neuron Add % Mean Params 
----------------------------------------------------------------------------------------------------
#1         34.0         12.8         30       100.0%       11.0        
#2         141.0        196.0        30       100.0%       17.4        
#3         265.7        177.2        28       100.0%       24.6        
#4         545.0        452.2        24       100.0%       31.1        
#5         865.5        505.5        20       100.0%       35.9        
#6         932.7        415.8        11       100.0%       41.8        
#7         1155.0       291.2        4        100.0%       45.8        
#8         1650.0       0.0          1        100.0%       57.